In [120]:
import pandas as pd
PATH_HAIKU = '../data/m_haiku.json'
df = pd.read_json(PATH_HAIKU)

In [123]:
len(df["id"].unique())

34831

In [137]:
df[df.duplicated(subset='id', keep="last")].head(200)

,id,content,author,data_from
2939,21699,児等の顔映して消えししやぼん玉,伊藤和,haiku-data.jp
2940,21697,朧夜の面影橋を渡りけり,伊藤和,haiku-data.jp
2941,21700,碧空をきりりと裂きて初燕,伊藤和,haiku-data.jp
2942,21698,老鶯の声も緑に美術館,伊藤和,haiku-data.jp
2943,21701,花筏よろめきあひて水の旅,伊藤和,haiku-data.jp
...,...,...,...,...
33607,4301,桃冷す水しろがねにうごきけり,百合山羽公,haiku-data.jp
33608,4300,海道を好みて走るいなびかり,百合山羽公,haiku-data.jp
33609,39939,花盗人ちりくる花を仰ぎけり,百合山羽公,haiku-data.jp
33610,39941,茶の花のかげのきてゐる囮かな,百合山羽公,haiku-data.jp


In [134]:
df[df["id"] == 21699]

,id,content,author,data_from
2939,21699,児等の顔映して消えししやぼん玉,伊藤和,haiku-data.jp
2945,21699,児等の顔映して消えししやぼん玉,伊藤和子,haiku-data.jp


In [125]:

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34962 entries, 0 to 34961
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         34962 non-null  int64 
 1   content    34962 non-null  object
 2   author     34962 non-null  object
 3   data_from  34962 non-null  object
dtypes: int64(1), object(3)
memory usage: 1.1+ MB


In [48]:
from transformers import BertJapaneseTokenizer, BertModel


import torch
import numpy as np

MODEL_NAME = "sonoisa/sentence-bert-base-ja-mean-tokens-v2"
tokenizer = BertJapaneseTokenizer.from_pretrained(MODEL_NAME)
model = BertModel.from_pretrained(MODEL_NAME)

c:\workspace\hAIku\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [80]:
max_length = 256
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def sentence_to_vector(sentence):
  encoding = tokenizer(
    sentence,
    max_length = max_length,
    padding = 'max_length',
    truncation = True,
    return_tensors = 'pt'
  )
  encoding = {k: v.to(device) for k, v in encoding.items()}
  attention_mask = encoding['attention_mask']

  #文章ベクトルを計算
  with torch.no_grad():
    output = model(**encoding)
    last_hidden_state = output.last_hidden_state
    averaged_hidden_state =(last_hidden_state*attention_mask.unsqueeze(-1)).sum(1)/attention_mask.sum(1,keepdim=True) 

  return averaged_hidden_state

def calc_similarity(sentence_vector1):
  ret_func = lambda sentence_vector2: torch.nn.functional.cosine_similarity(sentence_vector1, sentence_vector2, dim=1).detach().cpu().numpy().copy()[0]
  return ret_func

In [84]:
df = df.head(100)
df["content_vector"] = df["content"].apply(sentence_to_vector)

In [81]:
haiku_konbu = "流れつくこんぶに何が書いてあるか"
vec_konbu = sentence_to_vector(haiku_konbu)


In [82]:
calc_similarity(vec_konbu)(df["content_vector"][0])

0.061894618

In [85]:
calc_func = calc_similarity(vec_konbu)

df["similarity"] = df["content_vector"].apply(calc_func)

In [86]:
df

,id,content,author,data_from,content_vector,similarity
0,12716,若草や杖の穴より湧く温泉,相生緑葉,haiku-data.jp,"[[tensor(0.3880), tensor(-0.5346), tensor(-1.7...",0.061895
1,1416,ふらふらと死にゐし風が起き上る,相生垣瓜人,haiku-data.jp,"[[tensor(0.9668), tensor(-0.0035), tensor(0.32...",0.221810
2,1417,わが宿のいささ群竹酔ふ日かも,相生垣瓜人,haiku-data.jp,"[[tensor(-0.1582), tensor(-0.7197), tensor(-0....",0.232927
3,1425,クリスマス佛は薄目し給へり,相生垣瓜人,haiku-data.jp,"[[tensor(-1.4064), tensor(0.7763), tensor(-1.7...",0.171127
4,1419,一団の年賀状にぞ襲はれし,相生垣瓜人,haiku-data.jp,"[[tensor(-0.4190), tensor(-0.0145), tensor(-0....",0.435895
...,...,...,...,...,...,...
95,30139,先生とその先生に岬東風,青木伊佐恵,haiku-data.jp,"[[tensor(0.4793), tensor(0.3892), tensor(-0.65...",0.105850
96,30140,四阿の会話二言雨遍路,青木伊佐恵,haiku-data.jp,"[[tensor(-1.2596), tensor(-0.5257), tensor(-0....",0.164757
97,30138,特攻の島けぶりおり芽木の島,青木伊佐恵,haiku-data.jp,"[[tensor(0.1202), tensor(0.1482), tensor(-0.64...",0.195129
98,30142,碧眼の少年と見る原爆樹,青木伊佐恵,haiku-data.jp,"[[tensor(-0.4134), tensor(-0.0790), tensor(-0....",0.082512


In [88]:
df.sort_values('similarity', ascending=False)

,id,content,author,data_from,content_vector,similarity
4,1419,一団の年賀状にぞ襲はれし,相生垣瓜人,haiku-data.jp,"[[tensor(-0.4190), tensor(-0.0145), tensor(-0....",0.435895
24,1410,隙間風その数条を熟知せり,相生垣瓜人,haiku-data.jp,"[[tensor(0.0624), tensor(0.6305), tensor(-0.00...",0.366566
6,1415,何物が蛾を装ひて入り来るや,相生垣瓜人,haiku-data.jp,"[[tensor(-0.7189), tensor(-0.1740), tensor(0.4...",0.357818
92,26169,氷紋をなぞった指の行方かな,青木章子,haiku-data.jp,"[[tensor(-0.3852), tensor(0.1366), tensor(0.40...",0.349299
79,29174,指切りの針いっぽんが朧なり,相本寿美子,haiku-data.jp,"[[tensor(0.0033), tensor(-0.7605), tensor(-0.4...",0.318885
...,...,...,...,...,...,...
51,19022,盆梅に一湾据えて漁夫の婚,會田渓泉,haiku-data.jp,"[[tensor(0.1379), tensor(-0.0225), tensor(-1.1...",0.015112
18,1409,梅雨明けぬ猫がまづ木に駈け上がる,相生垣瓜人,haiku-data.jp,"[[tensor(0.1205), tensor(0.5899), tensor(-0.25...",0.006559
5,1426,亡き母に米寿の春を贈られし,相生垣瓜人,haiku-data.jp,"[[tensor(0.6411), tensor(-0.7814), tensor(-0.0...",0.006354
88,14477,春の雲キリンの首がさびしがる,相吉香湖,haiku-data.jp,"[[tensor(0.0390), tensor(1.0330), tensor(-0.05...",-0.021235


In [90]:
import os
import pickle
with open(os.path.join("storage", "haiku"), mode='wb') as f:
  pickle.dump(df, f, protocol=2)

In [91]:

with open(os.path.join("storage", "haiku"), 'rb') as f:
  mydata2 = pickle.load(f)
mydata2

,id,content,author,data_from,content_vector,similarity
0,12716,若草や杖の穴より湧く温泉,相生緑葉,haiku-data.jp,"[[tensor(0.3880), tensor(-0.5346), tensor(-1.7...",0.061895
1,1416,ふらふらと死にゐし風が起き上る,相生垣瓜人,haiku-data.jp,"[[tensor(0.9668), tensor(-0.0035), tensor(0.32...",0.221810
2,1417,わが宿のいささ群竹酔ふ日かも,相生垣瓜人,haiku-data.jp,"[[tensor(-0.1582), tensor(-0.7197), tensor(-0....",0.232927
3,1425,クリスマス佛は薄目し給へり,相生垣瓜人,haiku-data.jp,"[[tensor(-1.4064), tensor(0.7763), tensor(-1.7...",0.171127
4,1419,一団の年賀状にぞ襲はれし,相生垣瓜人,haiku-data.jp,"[[tensor(-0.4190), tensor(-0.0145), tensor(-0....",0.435895
...,...,...,...,...,...,...
95,30139,先生とその先生に岬東風,青木伊佐恵,haiku-data.jp,"[[tensor(0.4793), tensor(0.3892), tensor(-0.65...",0.105850
96,30140,四阿の会話二言雨遍路,青木伊佐恵,haiku-data.jp,"[[tensor(-1.2596), tensor(-0.5257), tensor(-0....",0.164757
97,30138,特攻の島けぶりおり芽木の島,青木伊佐恵,haiku-data.jp,"[[tensor(0.1202), tensor(0.1482), tensor(-0.64...",0.195129
98,30142,碧眼の少年と見る原爆樹,青木伊佐恵,haiku-data.jp,"[[tensor(-0.4134), tensor(-0.0790), tensor(-0....",0.082512


In [105]:

with open(os.path.join("storage", "tokenizer.pkl"), mode='wb') as f:
  pickle.dump(tokenizer, f, protocol=2)

In [106]:

with open(os.path.join("storage", "model.pkl"), mode='wb') as f:
  pickle.dump(model, f, protocol=2)

In [107]:

with open(os.path.join("..", "data", "model.pkl"), 'rb') as f:
  model = pickle.load(f)
with open(os.path.join("..", "data", "tokenizer.pkl"), 'rb') as f:
  tokenizer = pickle.load(f)

In [97]:

with open(os.path.join("..", "data", "haiku_vector.pkl"), 'rb') as f:
  open_haiku_vector = pickle.load(f)
open_haiku_vector.head(10)

,id,content_vector
0,12716,"[[0.3880200684070587, -0.5346135497093201, -1...."
1,1416,"[[0.9667874574661255, -0.0034706180449575186, ..."
2,1417,"[[-0.1582280993461609, -0.7196993231773376, -0..."
3,1425,"[[-1.4064396619796753, 0.7762742638587952, -1...."
4,1419,"[[-0.41900748014450073, -0.014512510970234871,..."
5,1426,"[[0.6410695314407349, -0.7813975811004639, -0...."
6,1415,"[[-0.7188867330551147, -0.17399494349956512, 0..."
7,1420,"[[0.533454954624176, -0.09149239212274551, 0.3..."
8,1418,"[[-0.1615806519985199, 0.7083917856216431, 0.4..."
9,1408,"[[1.184794545173645, 0.9602559804916382, -0.95..."


In [98]:
open_haiku_vector["content_vector"][0]

[[0.3880200684070587,
  -0.5346135497093201,
  -1.7283945083618164,
  0.29024991393089294,
  0.3231339156627655,
  -0.8004710078239441,
  1.0436722040176392,
  -0.13755503296852112,
  -0.4383918046951294,
  -0.17729811370372772,
  0.48080864548683167,
  -0.020076552405953407,
  0.052380066365003586,
  0.5927050709724426,
  1.5720373392105103,
  0.14987507462501526,
  0.5794453024864197,
  0.2039673775434494,
  0.4256622791290283,
  0.7358224987983704,
  -0.029036492109298706,
  0.01783769764006138,
  -0.06384135037660599,
  0.2267109602689743,
  -0.013813957571983337,
  0.9159336090087891,
  -0.1927061825990677,
  -0.5525330305099487,
  0.44109055399894714,
  0.5030291676521301,
  -0.2667377293109894,
  -0.38371291756629944,
  0.3010387420654297,
  0.9484040141105652,
  0.14990437030792236,
  -0.4770948886871338,
  -0.5033867955207825,
  -0.15734614431858063,
  0.04777047038078308,
  -0.35067903995513916,
  0.002345845103263855,
  0.2525540292263031,
  -1.3388938903808594,
  -0.0369959

In [115]:
from torch import Tensor


max_length = 256
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
def sentence_to_vector(sentence):
  encoding = tokenizer(
    sentence,
    max_length = max_length,
    padding = 'max_length',
    truncation = True,
    return_tensors = 'pt'
  )
  encoding = {k: v.to(device) for k, v in encoding.items()}
  attention_mask = encoding['attention_mask']

  #文章ベクトルを計算
  with torch.no_grad():
    output = model(**encoding)
    last_hidden_state = output.last_hidden_state
    averaged_hidden_state =(last_hidden_state*attention_mask.unsqueeze(-1)).sum(1)/attention_mask.sum(1,keepdim=True) 

  return averaged_hidden_state

def calc_similarity(selected_haiku_vectors: list[Tensor]):
  def ret_func(target: list[list[float]]):
    similarities:list[float] = [torch.nn.functional.cosine_similarity(haiku_vector, torch.tensor(target), dim=1).detach().numpy().copy()[0] for haiku_vector in selected_haiku_vectors]
    return sum(similarities) / len(similarities)
  return ret_func

In [118]:
vec_konbu = sentence_to_vector("流れつくこんぶに何が書いてあるか")
vec_mimizu = sentence_to_vector("私より彼女が綺麗糸みみず")
calc_konbu = calc_similarity([vec_konbu, vec_mimizu])
open_haiku_vector["similarity"] = open_haiku_vector["content_vector"].apply(calc_konbu)
open_haiku_vector.sort_values('similarity', ascending=False)

,id,content_vector,similarity
1942,8697,"[[-1.1777745485305786, -0.7245752811431885, 0....",0.636742
31329,17857,"[[-0.23273643851280212, 0.016235461458563805, ...",0.536933
34233,16259,"[[-0.40075916051864624, 0.3729219436645508, 0....",0.507121
6360,32724,"[[-0.4002126455307007, 0.4062024652957916, -0....",0.502155
12220,23588,"[[-0.7139098048210144, -0.7104634642601013, 0....",0.488308
...,...,...,...
16937,18530,"[[1.1444756984710693, 0.9872925281524658, -0.7...",-0.082188
2001,4207,"[[-0.15254409611225128, -0.09699451923370361, ...",-0.082926
31628,7840,"[[-0.9630804657936096, 0.11827445030212402, 0....",-0.087551
7515,32302,"[[0.6961217522621155, 0.004624891094863415, -1...",-0.097037
